# 📥 NYC Yellow Taxi Analytics — Landing Zone Ingestion
---
### 👤 Developer Profile
* **Name:** `cacciari@gmail.com`
* **Contact:** [cacciari@gmail.com](mailto:cacciari@gmail.com)
* **LinkedIn:** [linkedin.com/in/cacciari](https://linkedin.com/in/cacciari)

### 📋 Module Overview
* **File Name:** `01_ingestion_to_landing_zone.py`
* **Target Layer:** `Landing Zone (Staging Area)`
* **Short Description:** External data extraction engine that fetches historical `.parquet` trip data matrices from the NYC TLC cloud infrastructure. It acts as a batch collector, establishing transactional consistency on physical volumes prior to DB entry.

#### Imports

In [0]:
import urllib.request
import sys
import os

sys.path.append(os.path.abspath('..'))

#### Pipeline configuration

In [0]:
# PIPELINE CONFIGURATION
from constants import TARGET_YEAR, TARGET_MONTHS_NUMERIC, BASE_URL, VOLUME_PATH
# TARGET_YEAR = "2023"
# TARGET_MONTHS_NUMERIC = ["01", "02", "03", "04", "05", "06"]
# BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
# VOLUME_PATH= "/Volumes/ifood/lake/landing_zone"

#### Ingestion execution

* For this challenge, I'm considering _late arrivals_: data from analysis window made available after it ends. 
* I will consider that every travel that _begins_ in a specific month, belongs to that month.
    * According to this rule, travels beginning in December and ending in January (wrapping midnight) are not included as they belong to december. However, travels beginning in May and ending in June (wrapping midnight) are included as they begin in May.
* Including June for extra checks, ommiting December from prior year

In [0]:
print("Parameter Set:")
print(f"\tTarget Year: {TARGET_YEAR}")
print(f"\tTarget Months: {TARGET_MONTHS_NUMERIC}")
print(f"\tBase URL: {BASE_URL}")
print(f"\tVolume Path: {VOLUME_PATH}")

print("Starting ingestion to Landing Zone")

successful_downloads = []
failed_downloads = []

for month in TARGET_MONTHS_NUMERIC:
  file_name = f"yellow_tripdata_{TARGET_YEAR}-{month}.parquet"
  file_path = f"{BASE_URL}/{file_name}"
  volume_path = f"{VOLUME_PATH}/{file_name}"
  
  try:
      print(f"Processing {TARGET_YEAR}/{month}")
      urllib.request.urlretrieve(file_path, volume_path)
      print(f"File {file_name} downloaded to {volume_path}")
      successful_downloads.append(file_name)

  except Exception as e:
      print(f"Error downloading file {file_name} to {volume_path}. Reason: {str(e)}")
      failed_downloads.append(file_name)
            
print("Ingestion to Landing Zone completed.")

print(f"\tSuccessful Downloads:")
if len(successful_downloads) > 0:
    for files in successful_downloads:
        print(f"\t\t{files}")
else:
    print("\t\tNone")

print(f"\tFailed Downloads:")
if len(failed_downloads) > 0:
    for files in failed_downloads:
        print(f"\t\t{files}")
else:
    print("\t\tNone")    


In [0]:
print("Displaying Files in Volume Path:")
display(dbutils.fs.ls(VOLUME_PATH))